In [24]:
import pandas as pd
import os

event_df = pd.read_csv('../../Our Datasets/classifier_results_logreg.csv')

event_df['source_file'] = event_df['source_file'].astype(int)

match_ids = event_df['source_file'].unique()
match_ids

array([1886347, 1899585])

In [25]:
event_df = event_df.rename(columns={'Unique ID': 'unique_poss_id'})

event_df['source_file'] = event_df['source_file'].astype(str)

In [26]:
event_df

,row_index,true_value,predicted_class,predicted_probability,unique_poss_id,match_id,event_index,frame_start,frame_end,rec_player_id,rec_team_short,rec_team_id,dist_to_attacking_goal,team_out_of_possession_phase_type,player_targeted_dangerous,event_row_index,source_file,is_drawing,is_middle_third
0,0,0,0,0.196989,1886347_42_46,1886347.0,42.0,387.0,430.0,735578.0,Newcastle,1805.0,99.575170,high_block,0.0,0,1886347,1,0
1,3,0,1,0.818607,1886347_170_193,1886347.0,170.0,1492.0,1492.0,735574.0,Newcastle,1805.0,34.427403,chaotic/disruption,0.0,3,1886347,1,0
2,14,1,1,0.524581,1886347_452_466,1886347.0,452.0,3799.0,3799.0,50978.0,Newcastle,1805.0,51.188596,chaotic/disruption,1.0,14,1886347,1,1
3,19,1,1,0.553058,1886347_452_466,1886347.0,466.0,3854.0,3866.0,38673.0,Auckland FC,4177.0,47.054798,chaotic/disruption,1.0,19,1886347,1,1
4,20,1,1,0.555331,1886347_536_558,1886347.0,536.0,4452.0,4452.0,795507.0,Newcastle,1805.0,72.213891,medium_block,1.0,20,1886347,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,5357,0,0,0.361377,1899585_3944_3944,1899585.0,3944.0,51294.0,51308.0,23909.0,Wellington P FC,867.0,73.374887,chaotic/disruption,0.0,5357,1899585,0,0
118,5359,0,0,0.323110,1899585_4022_4022,1899585.0,4022.0,51826.0,51837.0,23418.0,Auckland FC,4177.0,46.164589,chaotic/disruption,0.0,5359,1899585,0,1
119,5362,0,1,0.596813,1899585_4066_4066,1899585.0,4066.0,52781.0,52781.0,23418.0,Auckland FC,4177.0,39.101266,chaotic/disruption,0.0,5362,1899585,0,0
120,5365,0,0,0.305413,1899585_4235_4235,1899585.0,4235.0,54418.0,54426.0,799091.0,Wellington P FC,867.0,82.264328,chaotic/disruption,0.0,5365,1899585,0,0


In [27]:
type(event_df.loc[1, 'frame_end'])

numpy.float64

This should take 7 minutes roughly to run

In [28]:
base_dir = "../../data/matches"
all_tracking = []
for mid in match_ids:
    folder = os.path.join(base_dir, str(mid))
    files = os.listdir(folder)
    track_files = [f for f in files if "tracking_extrapolated" in f.lower()]
    if not track_files:
        print(f"❌ No tracking_extrapolation file for match {mid}")
        continue
    file_path = os.path.join(folder, track_files[0])
    print(f"📄 Reading JSONL: {file_path}")
    tracking_df_temp = pd.read_json(file_path, lines=True)
    tracking_df_temp["match_id"] = mid
    tracking_df_temp['match_id'] = tracking_df_temp['match_id'].astype(str)
    tracking_df_temp['frame'] = tracking_df_temp['frame'].astype(int)
    merged = tracking_df_temp.merge(
        event_df[['source_file', 'unique_poss_id', 'frame_start', 'frame_end']],
        left_on='match_id',
        right_on='source_file',
        how='left'
    )
    merged['frame_start'] = merged['frame_start'].astype('Int64')
    merged['frame_end'] = merged['frame_end'].astype('Int64')

    merged['frame_start_rough_bound'] = merged['frame_start'] - 100
    merged['frame_end_rough_bound'] = merged['frame_end'] + 100

    mask = merged['frame_start'].notna() & merged['frame_end'].notna()

    filtered_tracking = merged[
        mask
        & merged['frame'].between(merged['frame_start_rough_bound'], merged['frame_end_rough_bound'])
    ].copy()
    filtered_tracking = pd.json_normalize(
        filtered_tracking.to_dict("records"),
        "player_data",
        ["frame", "timestamp", "period", "possession", "ball_data", 'frame_start', 'frame_end', 'frame_start_rough_bound', 'frame_end_rough_bound']
    )
    filtered_tracking["possession_player_id"] = filtered_tracking["possession"].apply(lambda x: x.get("player_id"))
    filtered_tracking["possession_group"] = filtered_tracking["possession"].apply(lambda x: x.get("group"))
    filtered_tracking[["ball_x", "ball_y", "ball_z", "is_detected_ball"]] = pd.json_normalize(filtered_tracking.ball_data)
    filtered_tracking["match_id"] = mid
    filtered_tracking['match_id'] = filtered_tracking['match_id'].astype(str)
    all_tracking.append(filtered_tracking)
tracking_all = pd.concat(all_tracking, ignore_index=True)
tracking_all = tracking_all.drop(columns=["possession", "ball_data"])

📄 Reading JSONL: ../../data/matches\1886347\1886347_tracking_extrapolated.jsonl
📄 Reading JSONL: ../../data/matches\1899585\1899585_tracking_extrapolated.jsonl


In [29]:
tracking_all

,x,y,player_id,is_detected,frame,timestamp,period,frame_start,frame_end,frame_start_rough_bound,frame_end_rough_bound,possession_player_id,possession_group,ball_x,ball_y,ball_z,is_detected_ball,match_id
0,-36.97,0.95,51009,False,183,2026-01-01 00:00:17.300,1.0,283,299,183,399,NaN,None,22.14,8.96,0.38,False,1886347
1,-8.82,-0.90,176224,False,183,2026-01-01 00:00:17.300,1.0,283,299,183,399,NaN,None,22.14,8.96,0.38,False,1886347
2,-9.34,8.22,51649,False,183,2026-01-01 00:00:17.300,1.0,283,299,183,399,NaN,None,22.14,8.96,0.38,False,1886347
3,-3.44,-10.49,50983,False,183,2026-01-01 00:00:17.300,1.0,283,299,183,399,NaN,None,22.14,8.96,0.38,False,1886347
4,-8.52,18.15,735578,False,183,2026-01-01 00:00:17.300,1.0,283,299,183,399,NaN,None,22.14,8.96,0.38,False,1886347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
562931,-19.23,3.13,51015,False,59327,2026-01-01 01:34:15.700,2.0,59199,59227,59099,59327,NaN,None,-52.19,13.29,0.04,True,1899585
562932,-52.78,12.66,799091,True,59327,2026-01-01 01:34:15.700,2.0,59199,59227,59099,59327,NaN,None,-52.19,13.29,0.04,True,1899585
562933,-23.45,-1.30,23909,False,59327,2026-01-01 01:34:15.700,2.0,59199,59227,59099,59327,NaN,None,-52.19,13.29,0.04,True,1899585
562934,-45.68,-0.80,159938,True,59327,2026-01-01 01:34:15.700,2.0,59199,59227,59099,59327,NaN,None,-52.19,13.29,0.04,True,1899585


In [30]:
tracking_all.to_csv('../../Our Datasets/processed_tracking_data_for_test_match_ids.csv', index=False)